# 02 PD Modeling and Validation

### Objective
Compare feature specifications and model families, evaluate calibration, and highlight out-of-time model risk.

### Inputs
- `reports/tables/pd_model_comparison_readme_v1.csv`
- `reports/tables/pd_model_feature_set_comparison_v1.csv`
- `reports/tables/pd_validation_metrics_v1.csv`
- `reports/tables/pd_holdout_vs_oot_v1.csv`
- `models/pd_coefficients.csv`

### Methodology
The project compares borrower-only and platform-informed feature sets, evaluates linear and nonlinear classifiers, and separates discrimination from probability calibration.

### Key Outputs
- Model comparison table
- Embedded underwriting signal ablation
- Holdout vs out-of-time validation
- Coefficient interpretation

In [1]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

reports = project_root / 'reports'
tables = reports / 'tables'
processed = project_root / 'data' / 'processed'

model_comparison = pd.read_csv(tables / 'pd_model_comparison_readme_v1.csv')
feature_sets = pd.read_csv(tables / 'pd_model_feature_set_comparison_v1.csv')
validation = pd.read_csv(tables / 'pd_validation_metrics_v1.csv')
holdout_vs_oot = pd.read_csv(tables / 'pd_holdout_vs_oot_v1.csv')
coefs = pd.read_csv(project_root / 'models' / 'pd_coefficients.csv')

In [2]:
model_comparison

,model_feature_set,roc_auc,pr_auc,brier_score,interpretation
0,Platform-informed logistic regression,0.709353,0.370143,0.145302,Strongest platform-informed baseline specifica...
1,Borrower-only logistic regression,0.674683,0.338786,0.149498,Measures borrower signal without embedded lend...
2,L2 logistic regression,0.709321,0.370149,0.145307,Retains performance with regularization and in...
3,Gradient boosting,0.706839,0.369964,0.145631,No material nonlinear improvement
4,Random forest,0.705018,0.367560,0.146269,No material improvement over logistic alternat...


In [3]:
feature_sets[['model_version', 'feature_set_description', 'roc_auc', 'pr_auc', 'brier_score', 'highest_lowest_decile_default_rate_ratio']]

,model_version,feature_set_description,roc_auc,pr_auc,brier_score,highest_lowest_decile_default_rate_ratio
0,A_baseline,"Baseline features with grade, sub_grade, and i...",0.709353,0.370143,0.145302,10.516971
1,B_remove_sub_grade,"Remove sub_grade, keep grade and int_rate",0.707409,0.368622,0.145540,9.660786
2,C_remove_grade_and_sub_grade,"Remove grade and sub_grade, keep int_rate and ...",0.705058,0.364693,0.146211,9.506773
3,D_borrower_only,Borrower and loan characteristics only; remove...,0.674683,0.338786,0.149498,6.077378


In [4]:
holdout_vs_oot

,metric,random_holdout,out_of_time
0,ROC-AUC,0.708443,0.695314
1,PR-AUC,0.368950,0.368088
2,Brier Score,0.145459,0.157900
3,Log Loss,0.455280,0.487525
4,Gini,0.416886,0.390627
5,KS Statistic,0.302722,NaN
6,Mean Predicted PD,0.199726,0.193668
7,Observed Default Rate,0.199651,0.218196


In [5]:
validation

,sample,roc_auc,roc_auc_ci_lower_95,roc_auc_ci_upper_95,pr_auc_average_precision,brier_score,log_loss,gini,ks_statistic,mean_predicted_pd,observed_default_rate
0,random_holdout,0.708443,0.706254,0.710754,0.368950,0.145459,0.455280,0.416886,0.302722,0.199726,0.199651
1,out_of_time,0.695314,NaN,NaN,0.368088,0.157900,0.487525,0.390627,NaN,0.193668,0.218196


In [6]:
coefs.head(10)

,feature,coefficient,odds_ratio,risk_direction,absolute_coefficient
0,categorical__grade_G,0.672295,1.958728,associated_with_higher_estimated_default_proba...,0.672295
1,categorical__grade_F,0.541801,1.719101,associated_with_higher_estimated_default_proba...,0.541801
2,categorical__addr_state_MS,0.432529,1.541150,associated_with_higher_estimated_default_proba...,0.432529
3,categorical__grade_E,0.334270,1.396920,associated_with_higher_estimated_default_proba...,0.334270
4,categorical__addr_state_AR,0.316396,1.372173,associated_with_higher_estimated_default_proba...,0.316396
5,categorical__addr_state_NE,0.291514,1.338452,associated_with_higher_estimated_default_proba...,0.291514
6,categorical__addr_state_OK,0.268077,1.307447,associated_with_higher_estimated_default_proba...,0.268077
7,categorical__purpose_small_business,0.243166,1.275281,associated_with_higher_estimated_default_proba...,0.243166
8,categorical__addr_state_LA,0.236397,1.266677,associated_with_higher_estimated_default_proba...,0.236397
9,categorical__addr_state_AL,0.218364,1.244040,associated_with_higher_estimated_default_proba...,0.218364


### Findings
- L2 logistic regression retained nearly all performance of the best baseline while preserving interpretability.
- Removing LendingClub grade, sub-grade, and interest rate reduced ROC-AUC from about 0.709 to about 0.675.
- Out-of-time validation showed temporal calibration drift: average predicted PD was below the observed default rate in the newer cohort.

### Limitations
- Coefficients are associations with estimated default probability, not causal effects.
- Out-of-time KS was not stored in the project artifacts and would require storing time-split predictions.